In [ ]:
import pandas as pd
from sequences.get_sequences import get_corners_sequences, get_sequences_by_possession
from tqdm.notebook import tqdm_notebook
import os
from corners.metric import evaluate_corner

In [ ]:
params = {
    'pass_to_zone_I': 0.0,
    'pass_to_zone_II': 0.2,
    'pass_to_zone_III': 0.3,
    'pass_incomplete': -0.1,
    'pass_shot_assist': 1,
    'pass_goal_assist': 1.5,
    'pass_cross': 1.2,
    'pass_by_opponent': -0.1,
    'shot_goal': 3,
    'shot_saved': 2,
    'shot_blocked': 0.5,
    'shot_off_target': 0.8,
    'shot_post': 2,
    'shot_saved_to_post': 2,
    'shot_saved_off_target': 1.5,
    'clearance_by_opponent': -0.1,
    'ball_recovery_by_opponent': -0.2,
    'ball_recovery_by_own_team': 1,
    'foul_committed_by_own_team': -0.2,
    'foul_committed_by_own_team_with_card': -0.3,
    'foul_committed_by_opponent': 0.5,
    'foul_committed_by_opponent_with_penalty': 1,
}

In [ ]:
def get_spring_round(df, df_core):
    df_spring = []
    spring_matches = df_core[esa_22_23_core['match_week'] >= 18]['match_id'].values
    for match_id in spring_matches:
        df_result = df[(df['match_id'] == match_id)]
        df_spring.append(df_result)
    df_spring = pd.concat(df_spring)
    return df_spring

In [ ]:
corners_with_scores = pd.read_pickle('corners_with_scores_example_final.pkl')
esa_22_23_core = pd.read_pickle('data/38_235.pkl')

In [ ]:
if os.path.exists('data/esa_22_23_matches_spring.pkl'):
    spring_events = pd.read_pickle('data/esa_22_23_matches_spring.pkl')
else:
    esa_22_23_matches = pd.read_pickle('data/esa_22_23_matches_preprocessed.pkl')
    spring_events = get_spring_round(esa_22_23_matches, esa_22_23_core)
    spring_events.to_pickle('data/esa_22_23_matches_spring.pkl')

In [ ]:
sequences = get_sequences_by_possession(spring_events)
corners = get_corners_sequences(sequences)

In [ ]:
corners = [evaluate_corner(corner, params) for corner in tqdm_notebook(corners, dynamic_ncols=True)]

In [ ]:
len(corners)

In [ ]:
corners[0]

In [ ]:
# create dict with number of types in corner.iloc[0][type]
corner_types = {}
for corner in corners:
    if corner.iloc[0]['type'] not in corner_types:
        corner_types[corner.iloc[0]['type']] = 1
    else:
        corner_types[corner.iloc[0]['type']] += 1 

In [ ]:
corner_types

In [ ]:
corners[0]

In [ ]:
final_corners_metric = pd.DataFrame()
for id, score_positive in tqdm_notebook(zip(corners_with_scores['id'].values, corners_with_scores['score_positive'].values), total=len(corners_with_scores)):
    start_id = id.split('#')[0]
    end_id = id.split('#')[1]
    for seq in corners:
        if seq['id'].values[0] == start_id and seq['id'].values[-1] == end_id:
            seq_with_metric = seq
            seq_with_metric['ssv_corner'] = len(seq) * [score_positive]
            seq_with_metric['seq_id'] = len(seq) * [id]
            dfs_result.append(seq_with_metric)
final_corners_with_metric = pd.concat(dfs_result)
final_corners_with_metric.to_pickle('fast_actions_ssv_1.pkl')
final_corners_with_metric

In [ ]:
len(final_corners_with_metric['seq_id'].unique())

In [ ]:
final_corners_with_metric.columns

In [ ]:
columns_to_drop = ['obv_for_after','obv_for_before','obv_against_after','obv_against_before', 'obv_against_net', 'obv_total_net', 'off_camera','related_events',
       'under_pressure', 'counterpress', 'out', 'start_area', 'end_area','area_code']

In [ ]:
final_corners_metric = final_corners_with_metric.drop(columns_to_drop, axis=1)

In [ ]:
final_corners_metric.rename(columns={'corner_score':'corner_score_event'}, inplace=True)
final_corners_metric.rename(columns={'score':'corner_score_seq'}, inplace=True)

In [ ]:
final_corners_metric.to_pickle('corners_spring_22_23_cs_ssv.pkl')

In [ ]:
fast_actions_ssv = pd.read_pickle('data/fast_actions_ssv.pkl')

In [ ]:
sequences_ssv_cs = pd.concat([fast_actions_ssv, final_corners_metric])

In [ ]:
len(sequences_ssv_cs['seq_id'].unique())

In [ ]:
sequences_ssv_cs.to_pickle('data/sequences_ssv_cs.pkl')

In [8]:
import pandas as pd

sequences_ssv_cs = pd.read_pickle('data/sequences_ssv_cs.pkl')

In [9]:
sequences_ssv_cs.columns

Index(['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type',
       'possession', 'possession_team', 'play_pattern', 'obv_for_net', 'team',
       'player', 'position', 'location', 'duration', 'event', 'match_id',
       'possession_team_id', 'player_id', 'context_locations',
       'teammate_locations', 'actor_locations', 'start_x', 'start_y', 'end_x',
       'end_y', 'start_area', 'end_area', 'ssv_fast_action', 'seq_id',
       'corner_score_event', 'corner_score_seq', 'ssv_corner'],
      dtype='object')

In [10]:
sequences_ssv_cs['play_pattern'].unique()

array(['From Goal Kick', 'Regular Play', 'From Keeper', 'From Throw In',
       'From Free Kick', 'From Counter', 'From Corner', 'Other',
       'From Kick Off'], dtype=object)

In [11]:
corners = sequences_ssv_cs[sequences_ssv_cs['play_pattern'] == 'From Corner']

In [20]:
fast_actions = sequences_ssv_cs[sequences_ssv_cs['play_pattern'] != 'From Corner']

In [12]:
len(corners)

65830

In [13]:
len(corners['seq_id'].unique())

1612

In [21]:

print(corners['ssv_corner'].min())
print(corners['ssv_corner'].max())

print(fast_actions['ssv_fast_action'].min())
print(fast_actions['ssv_fast_action'].max())

8.335523e-05
0.9752627
1.6423838e-08
0.99999845
